# 04 · Compare 18-class plant training runs

This notebook reads the Drive artifacts written by notebook 03 for the `v20261026_2355546` plant dataset. It compares EfficientNet-B2, DINOv3 ViT-S/16, and SigLIP2 Base where runs are available. It does not open the archive, train, or load model weights.

Runs must have identical dataset fingerprints and class mappings. Rank models by validation metrics; use the new internal test split as a final check. Identity accuracy is not a pet-toxicity score.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
DATASET_VERSION = 'v20261026_2355546'
SAVE_TO_DRIVE = True
IN_COLAB = 'google.colab' in sys.modules
if SAVE_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Use Colab for Drive storage, or set SAVE_TO_DRIVE=False locally.')
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_RUNS_DIR = Path('/content/drive/MyDrive/AI/meta-global-hackaton/runs')
else:
    BASE_RUNS_DIR = Path('./data/processed/runs').resolve()
EXPERIMENT_ID = f'plant_export_{DATASET_VERSION}_holdout10_seed{SEED}'
RUNS_DIR = BASE_RUNS_DIR / EXPERIMENT_ID
print('Reading:', RUNS_DIR)

## Load and validate saved runs

Each complete run has metrics, configuration, dataset metadata, class mapping, and test predictions. The dataset fingerprint covers archive member IDs, source labels, split membership, and image hashes. Runs from different archives or split versions cannot be compared here.

In [ ]:
def encoder_name(training: dict) -> str:
    model_name = training['model_name']
    if model_name == 'dinov3':
        return training['hf_dinov3_id']
    if model_name == 'siglip2':
        return training['hf_siglip2_id']
    return 'torchvision EfficientNet-B2'


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


required = ('metrics.json', 'config.json', 'dataset_metadata.json', 'class_mapping.json')
runs = []
if not RUNS_DIR.is_dir():
    raise FileNotFoundError(f'Run directory not found: {RUNS_DIR}. Run notebook 03 first.')
for run_dir in sorted(RUNS_DIR.iterdir()):
    if not run_dir.is_dir() or not all((run_dir / name).is_file() for name in required):
        continue
    metrics = read_json(run_dir / 'metrics.json')
    training = read_json(run_dir / 'config.json')['training']
    dataset = read_json(run_dir / 'dataset_metadata.json')
    mapping = read_json(run_dir / 'class_mapping.json')
    per_class_path = run_dir / 'test_per_class_metrics.csv'
    matrix_path = run_dir / 'test_confusion_matrix.csv'
    runs.append({
        'run': run_dir.name, 'model': training['model_name'],
        'encoder': encoder_name(training), 'seed': training['seed'],
        'source_version': training['source_version'],
        'epochs': training['epochs'], 'learning_rate': training['learning_rate'],
        'dataset_version': dataset['version_tag'],
        'dataset_fingerprint': dataset['dataset_fingerprint'],
        'class_mapping': mapping['class_to_idx'],
        'class_names': mapping['index_to_class'],
        'best_epoch': metrics['best_epoch'],
        'best_val_accuracy': metrics['best_val_accuracy'],
        'best_val_loss': metrics['best_val_loss'],
        'test_accuracy': metrics['test']['accuracy'],
        'test_macro_f1': metrics['test']['macro_f1'],
        'test_count': metrics['test']['count'],
        'history': metrics['history'],
        'per_class': pd.read_csv(per_class_path) if per_class_path.exists() else None,
        'confusion_matrix': pd.read_csv(matrix_path, index_col=0).to_numpy(dtype=int)
                            if matrix_path.exists() else None,
    })
if not runs:
    raise ValueError(f'No complete plant export runs found in {RUNS_DIR}')
if any(run['dataset_version'] != DATASET_VERSION for run in runs):
    raise ValueError('Dataset versions differ; compare these runs separately.')
if len({run['dataset_fingerprint'] for run in runs}) != 1:
    raise ValueError('Image contents or split membership differ; compare these runs separately.')
if len({json.dumps(run['class_mapping'], sort_keys=True) for run in runs}) != 1:
    raise ValueError('Class mappings differ; compare these runs separately.')
print('Comparable runs:', len(runs), '| Fingerprint:', runs[0]['dataset_fingerprint'])

## Compare validation performance

The table is sorted by best validation accuracy, then validation loss. The new internal test scores are displayed for transparency and should not be used to select a winner. Look at training curves for overfitting, especially for classes with few validation images.

In [ ]:
columns = ['run', 'model', 'encoder', 'seed', 'source_version', 'epochs',
           'learning_rate', 'best_epoch', 'best_val_accuracy', 'best_val_loss',
           'test_accuracy', 'test_macro_f1', 'test_count']
comparison = pd.DataFrame(runs)[columns].sort_values(
    ['best_val_accuracy', 'best_val_loss'], ascending=[False, True]
).reset_index(drop=True)
display(comparison.style.format({
    'learning_rate': '{:.1e}', 'best_val_accuracy': '{:.1%}',
    'best_val_loss': '{:.3f}', 'test_accuracy': '{:.1%}',
    'test_macro_f1': '{:.1%}',
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run in runs:
    history = pd.DataFrame(run['history'])
    axes[0].plot(history.epoch, history.train_accuracy, '--', alpha=0.6,
                 label=f"{run['run']} train")
    axes[0].plot(history.epoch, history.val_accuracy, marker='o',
                 label=f"{run['run']} val")
    axes[1].plot(history.epoch, history.train_loss, '--', alpha=0.6,
                 label=f"{run['run']} train")
    axes[1].plot(history.epoch, history.val_loss, marker='o',
                 label=f"{run['run']} val")
axes[0].set(title='Accuracy by epoch', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1))
axes[1].set(title='Cross-entropy loss by epoch', xlabel='Epoch', ylabel='Loss')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Per-class diagnostics and selected-run test view

The heatmap shows per-class test F1 for available runs. The confusion matrix is shown for the run selected by validation performance. Classes with only three test images have especially unstable estimates.

In [ ]:
per_class_frames = []
for run in runs:
    if run['per_class'] is not None:
        frame = run['per_class'].copy()
        frame['run'] = run['run']
        per_class_frames.append(frame)
if per_class_frames:
    per_class = pd.concat(per_class_frames, ignore_index=True)
    f1_table = per_class.pivot(index='class_name', columns='run', values='f1_score')
    f1_table = f1_table.reindex(runs[0]['class_names'])
    fig, ax = plt.subplots(figsize=(max(7, len(runs) * 2), 9))
    image = ax.imshow(f1_table.to_numpy(), vmin=0, vmax=1, cmap='YlGnBu', aspect='auto')
    ax.set(xticks=range(len(f1_table.columns)), xticklabels=f1_table.columns,
           yticks=range(len(f1_table.index)), yticklabels=f1_table.index,
           title='Test F1 by flower class and run')
    ax.tick_params(axis='x', rotation=45)
    fig.colorbar(image, ax=ax, label='F1 score')
    plt.tight_layout()
    plt.show()
else:
    print('No per-class CSVs found.')

winner = sorted(runs, key=lambda run: (-run['best_val_accuracy'], run['best_val_loss']))[0]
print('Selected by validation:', winner['run'])
print(f"Test accuracy: {winner['test_accuracy']:.1%} on {winner['test_count']} images")
if winner['confusion_matrix'] is not None:
    matrix = winner['confusion_matrix']
    class_names = winner['class_names']
    fig, ax = plt.subplots(figsize=(12, 10))
    image = ax.imshow(matrix, cmap='Blues')
    ax.set(xticks=range(len(class_names)), xticklabels=class_names,
           yticks=range(len(class_names)), yticklabels=class_names,
           xlabel='Predicted flower', ylabel='Actual flower',
           title=f"Test confusion matrix · {winner['run']}")
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()

## Interpretation and conclusion

- **Validation:** SigLIP2 and DINOv3 both reached 96.4% accuracy. SigLIP2 had the lower validation loss (0.144 vs. 0.213), so it was selected by the notebook's accuracy-then-loss rule. EfficientNet-B2 reached 85.6% validation accuracy with loss 0.494.
- **Internal test:** On 111 images, SigLIP2 scored 98.2% accuracy and 98.4% macro F1; DINOv3 scored 97.3% and 97.5%; EfficientNet-B2 scored 81.1% and 78.7%. These are single-seed results on this dataset split. Some classes have only three test images, so per-class estimates are imprecise.
- **Conclusion:** SigLIP2 is the strongest current candidate among these runs. DINOv3 is close; both transformer encoders outperformed EfficientNet-B2 on this split. Review the cross-split near-duplicate candidates found in notebook 01 before relying on the high test scores. Those candidates may inflate performance, but they have not been confirmed as duplicate photographs.
- This is a closed-set plant identity experiment on an internal split. It does not establish performance on independent real-world photos, identify plants outside the 18 classes, or assess pet toxicity.
